# RadImageNet .h5 → .pt Converter

**VERSION: radconvert-v1.0**

Converts Keras .h5 RadImageNet weights to PyTorch .pt format.
Run on Kaggle with the 'Pretrained RadImageNet' dataset attached.
Output: `/kaggle/working/RadImageNet-ResNet50.pt` (upload as dataset after).

In [ ]:
VERSION = "radconvert-v1.0"
print(f"VERSION: {VERSION}")

import os, glob
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import resnet50

# Find the .h5 file
h5_hits = glob.glob('/kaggle/input/**/RadImageNet-ResNet50_notop.h5', recursive=True)
assert h5_hits, "FATAL: RadImageNet-ResNet50_notop.h5 not found. Attach the 'Pretrained RadImageNet' dataset."
H5_PATH = h5_hits[0]
print(f"Found: {H5_PATH}")
print(f"Size: {os.path.getsize(H5_PATH) / 1e6:.1f} MB")


In [ ]:
# Load Keras weights via h5py (no TF needed)
import h5py

print("Reading .h5 structure...")
with h5py.File(H5_PATH, 'r') as f:
    def print_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: {obj.shape}")
    # List top-level groups
    print("Top-level keys:", list(f.keys())[:10])


In [ ]:
# Keras ResNet50 layer name mapping to PyTorch
# Keras: conv1_conv/kernel, bn1/gamma, etc.
# PyTorch: conv1.weight, bn1.weight, etc.

import h5py
import torch
from torchvision.models import resnet50
from collections import OrderedDict

print("Loading Keras weights and converting...")

pt_state = OrderedDict()

with h5py.File(H5_PATH, 'r') as f:
    # The h5 structure varies; try common patterns
    # Pattern 1: model_weights/<layer>/<param>
    # Pattern 2: <layer>/<param> directly
    
    def get_weight(path_variants):
        for p in path_variants:
            try:
                return f[p][()]
            except KeyError:
                continue
        return None
    
    # Try to find the model weights group
    root_keys = list(f.keys())
    print(f"Root keys: {root_keys}")
    
    # Common: 'model_weights' is the root
    wg = f['model_weights'] if 'model_weights' in f else f
    layer_names = list(wg.keys())
    print(f"Found {len(layer_names)} layers")
    print(f"Sample: {layer_names[:5]}")

print("\nDone reading structure. Manual mapping needed based on output above.")
print("If layer names look like 'conv1_conv', 'bn_conv1', etc., the auto-mapper below will work.")
